In [1]:
#### Load AI libraries:
# Load libraries supporting AI analysis (dependencies are managed with uv - see pyproject.toml)
import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown

load_dotenv() # Load environment variables from .env file

OPENAI_API_KEY  = os.getenv("openai_api_key")
#base_url = os.getenv("base_url")

In [2]:
#### Connect to the LLM
# Instead of downloading a model, we connect to one hosted on a university server
openai = OpenAI(api_key=OPENAI_API_KEY) # this is our main client

model_name = "gpt-4o-mini" # Can get this from lais dashboard or compute provider
# Change model back to gpt-5.6-luna 

print("--- Client Ready ---")

--- Client Ready ---


In [3]:
question = "What is the capital of France?" # this is your prompt

messages = [{"role": "user", "content": question}] # This is how prompts are structured

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content
display(Markdown(answer))

The capital of France is Paris.

## Data

In [8]:
#### Load raw Givealittle data
# Givealittle is New Zealand's largest crowdfunding platform - these are real,
# public health-related fundraising campaigns scraped into a spreadsheet.
# Unlike the Hansard transcripts (one plain text file per debate), this data is
# tabular: one row per campaign, with the free text we want to analyse
# (title, pitch, description, use of funds) sitting alongside structured fields
# like the amount raised and the number of donors.
campaigns = pd.read_excel("../data/givealittle/givealittle_health.xlsx")

print(f"Loaded {len(campaigns)} campaigns with {len(campaigns.columns)} columns:")
print(list(campaigns.columns))

Loaded 199 campaigns with 21 columns:
['title', 'pitch', 'hero', 'uri', 'amountRaised', 'goal', 'timeLeft', 'action', 'actionUri', 'start', 'end', 'n_questions', 'location', 'n_donors', 'description', 'use_of_funds', 'whos_involved', 'moderated', 'n_updates', 'updates', 'Q&A']


In [9]:
#### Check the data loaded correctly
# Peek at one campaign in full before we start analysing - the same habit as
# printing the first lines of a transcript: always look at your raw data first
campaign = campaigns.iloc[0]

print(f"Title:         {campaign['title']}")
print(f"Location:      {campaign['location']}")
print(f"Goal:          ${campaign['goal']:,.0f}")
print(f"Raised so far: ${campaign['amountRaised']:,.0f} from {campaign['n_donors']:.0f} donors")
print()
print("Description:")
print(campaign["description"])

Title:         Support Emma’s Family as They Cherish Every Moment
Location:      Otago
Goal:          $15,000
Raised so far: $9,920 from 129 donors

Description:
Emma is a loving Mum to 8 year old Sophie and Wife to Mark. She is one of the calmest, kindest souls you could ever meet. Even in the face of unimaginable challenges, her gentle strength continues to inspire everyone around her.
In July 2024, Emma was diagnosed with Stage 4 breast cancer. Despite always being vigilant about her health and never missing a mammogram, everything changed unexpectedly. Just seven months after her last scan, Emma visited her GP with what she thought was a sprained rib — only to learn her ribs were broken due to cancer that had already spread.
Since then, life has been a whirlwind of hospital trips, radiation, and chemotherapy in Dunedin as doctors have worked to hold off the cancer. In July this year, Emma required surgery after the cancer weakened her bone. When the initial treatments stopped worki

In [11]:
campaigns

,title,pitch,hero,uri,amountRaised,goal,timeLeft,action,actionUri,start,...,n_questions,location,n_donors,description,use_of_funds,whos_involved,moderated,n_updates,updates,Q&A
0,Support Emma’s Family as They Cherish Every Mo...,Supporting Emma and Her Family Through Stage 4...,https://static.givealittle.co.nz/assets/hero/8...,https://givealittle.co.nz/cause/supportforemma,9920.0,15000,89 days to go,Donate,https://givealittle.co.nz/donate/cause/support...,25 Oct 2025,...,0,Otago,129.0,Emma is a loving Mum to 8 year old Sophie and ...,All funds raised will go to Mark to support hi...,Created by_x000D_\nAbbey Cousens \n\n\n\n_x00...,Page Moderated,1.0,Thank You\n26 October 2025\nHuge thank you fro...,NaN
1,Southland Farm Family Facing Breast Cancer,Help ease Tammy and Trent's financial pressure...,https://static.givealittle.co.nz/assets/hero/c...,https://givealittle.co.nz/cause/southland-farm...,3730.0,0,178 days to go,Donate,https://givealittle.co.nz/donate/cause/southla...,24 Oct 2025,...,0,Southland,18.0,Tammy and Trent have always been the first to ...,Living expenses while unable to generate incom...,Created by_x000D_\nAdele Cotton \n\n\n\n_x000...,Page Moderated,NaN,NaN,NaN
2,Please support Cody and his Mum Mel on their c...,Please support Cody and his Mum Mel on their c...,https://static.givealittle.co.nz/assets/hero/b...,https://givealittle.co.nz/cause/please-support...,4453.0,0,488 days to go,Donate,https://givealittle.co.nz/donate/cause/please-...,24 Oct 2025,...,0,"Riverton, Southland",52.0,In September 2025 15-year-old Cody Gnys sudden...,All funds raised will go directly towards supp...,Created by_x000D_\nJulie McEwan-Franks \n\n\n...,Page Moderated,NaN,NaN,NaN
3,"“Together for Thomas — Backing a Father, Frien...",This Givealittle page is a heartfelt tribute t...,https://static.givealittle.co.nz/assets/hero/c...,https://givealittle.co.nz/cause/together-for-t...,845.0,10000,88 days to go,Donate,https://givealittle.co.nz/donate/cause/togethe...,24 Oct 2025,...,0,"Whitianga, Waikato",10.0,"Our Dad, Thomas, is the kindest soul and the b...",We’re raising funds to help Dad live as fully ...,Created by_x000D_\n _x000D_\n shar...,Page Moderated,NaN,NaN,NaN
4,Broken but not out yet,Hate asking for help,https://static.givealittle.co.nz/assets/hero/8...,https://givealittle.co.nz/cause/broken-but-not...,15.0,8000,343 days to go,Donate,https://givealittle.co.nz/donate/cause/broken-...,24 Oct 2025,...,0,"New Plymouth, Taranaki",3.0,My story is an interesting one was fit working...,Money will be spent on bills keeping afloat wh...,"Created by, and paying to a verified bank acco...",Page Moderated,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,Healing Trauma,A journey of a mother making changes to break ...,https://static.givealittle.co.nz/assets/hero/e...,https://givealittle.co.nz/cause/healing-trauma,190.0,4500,32 days to go,Donate,https://givealittle.co.nz/donate/cause/healing...,29 Aug 2025,...,0,"Whangarei, Northland",4.0,Kia ora.\nMy name is Heaven. I have experience...,Māori traditional therapy. The tattoo artist. ...,"Created by, and paying to a verified bank acco...",Page Moderated,NaN,NaN,NaN
195,"Help us land lifesaving support, in Hanmer Spr...",Hanmer Springs Helipad Project,https://static.givealittle.co.nz/assets/hero/0...,https://givealittle.co.nz/cause/help-us-land-l...,8495.0,50000,123 days to go,Donate,https://givealittle.co.nz/donate/cause/help-us...,29 Aug 2025,...,0,"Hanmer Springs, Canterbury",28.0,"When every minute counts, safe and fast access...",Hanmer Springs Helipad Project - Planning & co...,"Created by, and paying to a verified bank acco...",Page Moderated,NaN,NaN,NaN
196,Support for Andrew– A Friend in Need,"Our dear friend and neighbour, Andrew, has rec...",https://static.givealittle.co.nz/assets/hero/a...,https://givealittle.co.nz/cause/support-for-an...,16840.0,0,closed,Read more,https://givealittle.co.nz/cause/support-for-an...,29 Aug 2025,

In [12]:
#### The extraction prompt
# The same prompt works for every campaign, so we define it once as a constant.
# It asks for one JSON object per campaign - some fields come from the text,
# some (like smiling or image_type) can only come from the image.
extraction_prompt = """
The below message is text extracted from givealittle, a crowdfunding platform. It's a health related campaign.
I've also included the hero image for the campaign.
For the text below, extract the following information, in JSON format:
condition: the primary health condition mentioned in the text
ICD10: the ICD10 code for the primary health condition
ICD: the top level ICD chapter for the primary health condition. One of:
    Chapter A00-B99 - Certain infectious and parasitic diseases
    Chapter C00-D49 - Neoplasms
    Chapter D50-D89 - Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
    Chapter E00-E89 - Endocrine, nutritional and metabolic diseases
    Chapter F01-F99 - Mental, Behavioral and Neurodevelopmental disorders
    Chapter G00-G99 - Diseases of the nervous system
    Chapter H00-H59 - Diseases of the eye and adnexa
    Chapter H60-H95 - Diseases of the ear and mastoid process
    Chapter I00-I99 - Diseases of the circulatory system
    Chapter J00-J99 - Diseases of the respiratory system
    Chapter K00-K95 - Diseases of the digestive system
    Chapter L00-L99 - Diseases of the skin and subcutaneous tissue
    Chapter M00-M99 - Diseases of the musculoskeletal system and connective tissue
    Chapter N00-N99 - Diseases of the genitourinary system
    Chapter O00-O9A - Pregnancy, childbirth and the puerperium
    Chapter P00-P96 - Certain conditions originating in the perinatal period
    Chapter Q00-Q99 - Congenital malformations, deformations and chromosomal abnormalities
    Chapter R00-R99 - Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified
    Chapter S00-T88 - Injury, poisoning and certain other consequences of external causes
    Chapter V00-Y99 - External causes of morbidity
    Chapter Z00-Z99 - Factors influencing health status and contact with health services
    Chapter U00-U85 - Codes for special purposes
name: the name of the person this campaign is for
gender: the gender of the person this campaign is for, one of Male, Female or Other/unknown
age: the age of the person this campaign is for
age_group: the age group of the person this campaign is for, one of 0-14, 15-64, 65+ or indeterminate/unknown
ethnicity: the ethnicity of the person this campaign is for. If not mentioned in the text, guess their ethnicity from the image.
urgency: a number from 0-100, indicating how urgent the need is
sentiment: a number from 0-100, indicating the sentiment of the text, where 100 is the most positive, and 0 is the most negative
truth: a number from 0-100, indicating how truthful the text is, where 100 is the most truthful, and 0 is the least truthful
notes: any additional information about how you processed this text, such as warnings or errors
smiling: a boolean indicating whether the person in the image is smiling
deservingness: a number from 0-100, indicating how deserving the person is of receiving funds, where 100 is the most deserving, and 0 is the least deserving
attractiveness: a number from 0-100, indicating how attractive the person is, where 100 is the most attractive, and 0 is the least attractive
use: The main use of the raised funds - one or more (comma separated) of: medical expenses, experimental therapies, travel expenses, lost wages
region: The region in New Zealand where the person is located, one of: Northland, Auckland, Waikato, Bay of Plenty, Gisborne, Hawke's Bay, Taranaki, Manuwatū-Whanganui, Wellington, Tasman, Nelson, Marlborough, West Coast, Canterbury, Otago, Southland
narrative_clarity: a number from 0-100, indicating how clear the narrative is, where 100 is the most clear, and 0 is the least clear
narrative_quality: a number from 0-100, indicating how well written the narrative is
emotional_tone: grateful | desperate | hopeful | neutral | etc
image_type: selfie | portrait | symbolic | environment | group | other
face_visible: true | false
facial_expression: smiling | neutral | serious | emotional | not_detectable
image_quality: high | medium | low
progression: a number from 0-100, indicating how advanced the condition is, where 100 is the most advanced, and 0 is the least advanced
treatment: a number from 0-100, indicating how much treatment the person has received, where 100 is the most treatment, and 0 is the least treatment
treatment_effectiveness: a number from 0-100, indicating how effective the treatment has been, where 100 is the most effective, and 0 is the least effective
treatment_side_effects: a number from 0-100, indicating how severe the side effects of the treatment have been
site: If the campaign is for cancer, what is the primary cancer site?
stage: If the campaign is for cancer, what stage is the cancer at?
reason: Summarise how donated funds will be used and the reason for requesting donations

Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid.
"""

In [ ]:
#### Analyse every campaign - text and image together
# Same loop-and-collect pattern as the Hansard workflow. The only new idea:
# 'content' is now a LIST of parts (text, image, text) instead of one string.
from tqdm import tqdm

all_campaigns = [] # one extracted record per campaign ends up here

for row in tqdm(campaigns.itertuples(index=False), total=len(campaigns)):

    # The free text lives in several columns, so stitch them into one labelled
    # block - the model sees the same thing a donor reading the page would
    campaign_text = "\n\n".join([
        f"Title: {row.title}",
        f"Pitch: {row.pitch}",
        f"Description: {row.description}",
        f"Use of funds: {row.use_of_funds}",
    ])

    messages = [
        {
            "role": "user",
            "content": [
                # Part 1: the instructions
                {"type": "text", "text": extraction_prompt},
                # Part 2: the hero image - just the URL, the server fetches it
                {"type": "image_url", "image_url": {"url": row.hero}},
                # Part 3: the campaign text itself
                {"type": "text", "text": campaign_text},
            ]
        }
    ]

    # One campaign failing (e.g. a broken image URL) shouldn't kill the whole run,
    # so catch API errors, record which row failed, and carry on
    try:
        response = openai.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=0, # coding/extraction - we want reproducible output
            response_format={"type": "json_object"} # ask the server to only return valid JSON
        )
    except Exception as error:
        print(f"{row.uri}: API call failed ({error})")
        continue

    raw_output_string = response.choices[0].message.content.strip()

    try:
        result_json = json.loads(raw_output_string)
    except json.JSONDecodeError:
        print(f"{row.uri}: model did not return valid JSON. Raw output:\n{raw_output_string}\n")
        continue

    # Keep the campaign URL so results can be joined back to the original data
    result_json["uri"] = row.uri
    all_campaigns.append(result_json)

# Collect into a DataFrame and save alongside the other extracted outputs
campaigns_df = pd.DataFrame(all_campaigns)
campaigns_df.to_csv("../data/extracted/givealittle_extracted.csv", index=False)

print(f"Saved {len(campaigns_df)} campaigns to ../data/extracted/givealittle_extracted.csv")
campaigns_df